# MedQuad Medical RAG System (PoC)

Setting up a RAG (Retrieval-Augmented Generation) architecture on the MedQuad medical question-answering dataset using LangChain, ChromaDB, and OpenAI (LCEL).

In [2]:
import os
from langchain_community.document_loaders.csv_loader import CSVLoader
import random
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from openai import OpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
# Set the user agent for LangChain retrieval lab
os.environ["USER_AGENT"] = "LangChainRetrievalLab/1.0"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key)

## Loading the Dataset

The MedQuad dataset is a medical dataset in a Q&A format compiled from reliable U.S.health institutions such as  NIH, Cancer.gov and MedlinePlus, etc [1]. It contains 16,407 rows of question-answer pairs.

In [17]:
# Current Working Directory (Mevcut çalışma dizini) üzerinden göreceli yol oluşturma
base_dir = os.getcwd()
data_path = os.path.join(base_dir, "data", "medDataset_processed.csv")

# Load the CSV file using CSVLoader
csv_loader = CSVLoader(file_path=data_path, metadata_columns=["qtype"], encoding="utf-8")
# Load the documents from the CSV file
documents = csv_loader.load()

# Print the first document's content and metadata
print("Page Content:\n", documents[0].page_content)
print("Metadata:\n", documents[0].metadata)

Page Content:
 Question: Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?
Answer: LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.
Metadata:
 {'source': 'c:\\Users\\duygu\\Documents\\Projects\\RAG_PoC_Chunking\\data\\medDataset_processed.csv', 'row': 0, 'qtype': 'susceptibility'}


In [18]:
# Basic verification 
total_chars = sum(len(doc.page_content) for doc in documents)

print(f"Document loaded successfully.")
print(f"Total Records: {len(documents)}")
print(f"Total Character Count: {total_chars}")

Document loaded successfully.
Total Records: 16407
Total Character Count: 22529063


## Chunking & Sampling

In [19]:
# Split the documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,  # No overlap for Structured Q&A
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents into chunks
chunked_docs = text_splitter.split_documents(documents)

# Sample 10% of the chunked documents
sample_size = int(len(chunked_docs) * 0.10)
sampled_docs = chunked_docs[:sample_size]

print(f"Total Chunks: {len(chunked_docs)}")
print(f"Sampled %10: {len(sampled_docs)} chunk")

Total Chunks: 37700
Sampled %10: 3770 chunk


## Vector Store 

In [20]:
# Create embeddings for the documents using OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a Chroma vector store from the documents and embeddings
vectorstore = Chroma.from_documents(
    documents=sampled_docs,
    embedding=embeddings,
    collection_name="medquad_csvloader"
)

## Retriever

In [13]:
# Create a retriever from the vector store with a specified number of results to return
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## RAG Pipeline

In [21]:
prompt_template = """Answer the question using the medical context below.
If there is no answer in the context, say, "There is no information related to this topic in the provided dataset."

Context: {context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [24]:
# Format a list of documents into a single string
def format_docs(docs):
    '''Formats a list of documents into a single string.'''
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# LCEL Flow: Create a RAG chain that retrieves documents and generates an answer using the LLM
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm )

## Test

In [25]:
test_question = "What are the symptoms of Lymphocytic Choriomeningitis?"
print(f"Question: {test_question}\n")

response = rag_chain.invoke(test_question)

print("AI Response :")
print(response.content)

Question: What are the symptoms of Lymphocytic Choriomeningitis?

AI Response :
LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection without symptoms or mild febrile illnesses are more common clinical manifestations. 

For infected persons who do become ill, onset of symptoms usually occurs 8-13 days after exposure to the virus as part of a biphasic febrile illness. This initial phase, which may last as long as a week, typically begins with any or all of the following symptoms: fever, malaise, lack of appetite, muscle aches, headache, nausea, and vomiting. Other symptoms appearing less frequently include sore throat, cough, joint pain, chest pain, testicular pain, and parotid (salivary gland) pain.


## References

1. Ben Abacha, A., & Demner-Fushman, D. (2019). A Question-Entailment Approach to Question Answering. BMC Bioinformatics, 20(1), 511. https://doi.org/10.1186/s12859-019-3119-4